# 실습 1. 자료 탐색과 정규성 검정

사회약학 연구방법 노트의 '실습 1. 자료 탐색과 정규성 검정'에 딸린 노트북입니다. scikit-learn에 들어 있는 위스콘신 유방암 진단 자료(유방 종괴 세침흡인 검체 569건)로 요약통계와 그래프를 만들고, Table 1의 평균 ± SD, 중앙값(IQR), SMD를 계산하고, Q-Q plot과 Shapiro-Wilk 검정으로 분포를 확인한 뒤, 30개 변수를 한꺼번에 검정할 때의 다중비교 보정을 실행합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 1에 있습니다.


## a. 실습 데이터 준비

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤, scikit-learn에 들어 있는 위스콘신 유방암 진단 자료를 읽습니다. 인터넷 연결은 필요 없습니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.datasets import load_breast_cancer
import scipy, matplotlib, statsmodels, sklearn

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scipy       ", scipy.__version__)
print("matplotlib  ", matplotlib.__version__)
print("statsmodels ", statsmodels.__version__)
print("scikit-learn", sklearn.__version__)

### 셀 2. 데이터 불러오기

In [ ]:
bc = load_breast_cancer(as_frame=True)
df = bc.frame.copy()     # 설명변수 30개 + target 1개
print(df.shape)
print(df["target"].value_counts())

### 셀 3. 진단 변수를 읽기 쉽게 바꾸기

In [ ]:
labels = {0: "malignant", 1: "benign"}      # sklearn의 코딩
df["diagnosis"] = pd.Categorical(df["target"].map(labels),
                                 categories=["benign", "malignant"])
df["malignant"] = (df["target"] == 0).astype(int)   # 1 = 악성
print(df["malignant"].sum())           # 악성 건수
pd.crosstab(df["target"], df["diagnosis"])   # 바꾼 결과 확인

### 셀 4. 설명변수 30개의 구성

In [ ]:
features = bc.feature_names.tolist()   # 설명변수 30개의 이름
print(len(features))
print(features[:10])    # 0–9번: mean 계열
print(features[10:13])  # 10–19번: error 계열 (앞의 3개만)
print(features[20:23])  # 20–29번: worst 계열 (앞의 3개만)

### 셀 5. 앞의 5행

In [ ]:
df[["mean radius", "mean smoothness", "area error",
    "diagnosis", "malignant"]].head()

## b. 요약통계와 그래프

### 셀 6. describe()

In [ ]:
cols = ["mean radius", "mean smoothness", "mean area", "area error"]
df[cols].describe().round(3)

### 셀 7. 평균, 중앙값, 왜도

In [ ]:
summary = pd.DataFrame({
    "mean": df[cols].mean(),
    "median": df[cols].median(),
    "skewness": df[cols].apply(stats.skew),   # 열마다 왜도
})
summary.round(3)

### 셀 8. 히스토그램

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, col in zip(axes, ["mean smoothness", "area error"]):
    ax.hist(df[col], bins=30, color="lightgray", edgecolor="gray")
    ax.axvline(df[col].mean(), color="C0", label="Mean")
    ax.axvline(df[col].median(), color="C1", linestyle="--",
               label="Median")
    ax.set_title(col)
    ax.set_xlabel("Value")
    ax.set_ylabel("Number of samples")
    ax.legend()
plt.tight_layout()
plt.show()

### 셀 9. 진단별 요약

In [ ]:
df.groupby("diagnosis", observed=True)["mean radius"].agg(
    ["count", "mean", "std", "median"]).round(2)

### 셀 10. 진단별 상자그림

In [ ]:
groups = ["benign", "malignant"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, col in zip(axes, ["mean radius", "area error"]):
    data = [df.loc[df["diagnosis"] == g, col] for g in groups]
    ax.boxplot(data)
    ax.set_xticks([1, 2], groups)
    ax.set_title(col)
    ax.set_ylabel("Value")
plt.tight_layout()
plt.show()

### 셀 11. 군별 평균과 표준편차

In [ ]:
t1_vars = ["mean radius", "mean texture", "mean smoothness",
           "mean area", "area error"]
g = df.groupby("diagnosis", observed=True)[t1_vars]
m = g.mean().T     # 행: 변수, 열: benign / malignant
s = g.std().T      # 표준편차 (n - 1로 나눔)
pd.concat({"mean": m, "sd": s}, axis=1).round(3)

### 셀 12. 군별 사분위수

In [ ]:
q = g.quantile([0.25, 0.5, 0.75]).T   # 1사분위수, 중앙값, 3사분위수
q.round(2)

### 셀 13. 표준화 평균차(SMD)

In [ ]:
# SMD = (평균 차이) / 두 군 분산의 평균의 제곱근
smd = (m["malignant"] - m["benign"]) / np.sqrt(
    (s["malignant"] ** 2 + s["benign"] ** 2) / 2)
print(smd.round(2))

## c. Q-Q plot과 정규성 검정

### 셀 14. Q-Q plot: 양성 종양 안에서

In [ ]:
benign = df[df["diagnosis"] == "benign"]      # 양성 357건
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, col in zip(axes, ["mean radius", "area error"]):
    stats.probplot(benign[col], dist="norm", plot=ax)
    ax.set_title(col + " (benign)")
plt.tight_layout()
plt.show()

### 셀 15. Shapiro-Wilk 검정

In [ ]:
for col in ["mean radius", "area error"]:
    x = benign[col]
    w, p = stats.shapiro(x)       # 검정통계량 W와 p-value
    print(f"{col}: n = {len(x)}, skewness = {stats.skew(x):.2f}, "
          f"W = {w:.3f}, p = {p:.3g}")

### 셀 16. 표본이 클 때

In [ ]:
x = df["mean smoothness"]             # 전체 569건
w, p = stats.shapiro(x)
print(f"n = {len(x)}, skewness = {stats.skew(x):.2f}, "
      f"W = {w:.3f}, p = {p:.2g}")

fig, ax = plt.subplots(figsize=(4.5, 3.6))
stats.probplot(x, dist="norm", plot=ax)
ax.set_title("mean smoothness (all, n = 569)")
plt.show()

### 셀 17. 표본크기와 정규성 검정

In [ ]:
rng = np.random.default_rng(2026)      # 난수 시드 고정
xs = df["mean smoothness"].to_numpy()
for n in [20, 50, 100, 200, 400]:
    rejected = 0
    for i in range(1000):              # 표본을 1000번 뽑아 검정
        sample = rng.choice(xs, size=n, replace=False)
        if stats.shapiro(sample).pvalue < 0.05:
            rejected += 1
    print(f"n = {n:3d}: p < 0.05 in {rejected / 1000:.1%}")

## d. 다중비교 보정

### 셀 18. 30개 변수의 t 검정

In [ ]:
mal = df[df["diagnosis"] == "malignant"]
ben = df[df["diagnosis"] == "benign"]
res = stats.ttest_ind(mal[features], ben[features], equal_var=False)
tt = pd.DataFrame({"t": res.statistic, "p": res.pvalue},
                  index=features)
print((tt["p"] < 0.05).sum(), "of", len(tt), "features: p < 0.05")
tt.sort_values("p").head(3)

### 셀 19. 1장 c 절의 다섯 p값으로 확인하기

In [ ]:
from statsmodels.stats.multitest import multipletests

p5 = [0.004, 0.012, 0.021, 0.038, 0.260]   # 1장 c 절 표의 원래 p
for method in ["bonferroni", "holm", "fdr_bh"]:
    reject, p_adj, _, _ = multipletests(p5, alpha=0.05, method=method)
    print(f"{method:10s}: {p_adj.round(4)}  {reject.sum()} significant")

### 셀 20. Bonferroni, Holm, Benjamini-Hochberg

In [ ]:
for method in ["bonferroni", "holm", "fdr_bh"]:
    reject, p_adj, _, _ = multipletests(tt["p"], alpha=0.05,
                                        method=method)
    tt["p_" + method] = p_adj          # 보정 p-value를 새 열로
    print(f"{method:10s}: {reject.sum()} significant")

### 셀 21. 경계에 있는 변수들

In [ ]:
show_cols = ["p", "p_bonferroni", "p_holm", "p_fdr_bh"]
tt.sort_values("p")[show_cols].tail(6)   # p가 큰 쪽 6개

### 셀 22. 귀무가설이 참일 때 (한 번)

In [ ]:
# 양성 357건을 무작위로 둘로 나눔: 두 군은 같은 집단 (귀무가설이 참)
half_a = ben.sample(frac=0.5, random_state=3)
half_b = ben.drop(half_a.index)
p_null = stats.ttest_ind(half_a[features], half_b[features],
                         equal_var=False).pvalue
print(len(half_a), len(half_b))
print("no correction:", (p_null < 0.05).sum())
for method in ["bonferroni", "holm", "fdr_bh"]:
    print(f"{method}:", multipletests(p_null, method=method)[0].sum())

### 셀 23. 귀무가설이 참일 때 (1000번 반복)

In [ ]:
rows = []
for i in range(1000):                 # 무작위 분할을 1000번 반복
    a = ben.sample(frac=0.5, random_state=i)
    b = ben.drop(a.index)
    p = stats.ttest_ind(a[features], b[features],
                        equal_var=False).pvalue
    row = {"none": (p < 0.05).sum()}
    for method in ["bonferroni", "holm", "fdr_bh"]:
        row[method] = multipletests(p, method=method)[0].sum()
    rows.append(row)
sim = pd.DataFrame(rows)       # 1000행: 반복마다 거짓양성 수
pd.DataFrame({"FWER": (sim > 0).mean(),   # 1개 이상 나온 비율
              "mean_FP": sim.mean()})     # 평균 거짓양성 수

### 셀 24. 작은 표본에서 세 방법의 차이

In [ ]:
# 악성 10건, 양성 10건만 뽑은 작은 연구 (실제 차이는 있음)
small_m = mal.sample(10, random_state=0)
small_b = ben.sample(10, random_state=0)
p_small = stats.ttest_ind(small_m[features], small_b[features],
                          equal_var=False).pvalue
print("no correction:", (p_small < 0.05).sum())
for method in ["bonferroni", "holm", "fdr_bh"]:
    print(f"{method}:", multipletests(p_small, method=method)[0].sum())

## e. 과제 정답

**과제 1.** 셀 21의 표에서 fractal dimension error 행을 말로 옮기고, 이 행이 말해 주지 않는 것을 두 가지 이상 쓰세요. 두 군의 평균 차이와 그 95% 신뢰구간, SMD를 구해 셀 13의 mean radius와 비교하세요.

### 셀 25. 과제 1 정답. 경계에 있는 변수의 차이 크기

In [ ]:
v = "fractal dimension error"
r = stats.ttest_ind(mal[v], ben[v], equal_var=False)   # 셀 18과 같은 검정
ci = r.confidence_interval()                           # 평균 차이의 95% CI
diff = mal[v].mean() - ben[v].mean()
smd_v = diff / np.sqrt((mal[v].var() + ben[v].var()) / 2)
print(f"mean: malignant {mal[v].mean():.5f}, benign {ben[v].mean():.5f}")
print(f"difference {diff:.5f} (95% CI {ci.low:.5f} to {ci.high:.5f})")
print(f"difference / benign mean = {diff / ben[v].mean():.1%}")
print(f"SMD {smd_v:.2f}  (mean radius: {smd['mean radius']:.2f})")

**과제 2.** 셀 18의 Welch t 검정을 Mann-Whitney 검정(`stats.mannwhitneyu`)으로 바꿔 30개 변수를 다시 검정하고, Holm 보정 후 유의한 개수를 비교하세요. fractal dimension error와 symmetry error의 p가 어떻게 바뀌는지, 왜 그런지 두 군의 중앙값, 평균, SD, 최댓값으로 설명하세요.

### 셀 26. 과제 2 정답. Welch t 검정을 Mann-Whitney 검정으로 바꾸기

In [ ]:
tt["p_mw"] = [stats.mannwhitneyu(mal[f], ben[f]).pvalue
              for f in features]               # 30개 변수의 Mann-Whitney 검정
holm_mw = multipletests(tt["p_mw"], method="holm")[0]
print("Welch       : p < 0.05", (tt["p"] < 0.05).sum(),
      "| after Holm", (tt["p_holm"] < 0.05).sum())
print("Mann-Whitney: p < 0.05", (tt["p_mw"] < 0.05).sum(),
      "| after Holm", holm_mw.sum())
two = ["fractal dimension error", "symmetry error"]
for f in two:
    print(f"{f:23}: Welch p = {tt.loc[f, 'p']:.3f}, "
          f"Mann-Whitney p = {tt.loc[f, 'p_mw']:.2g}")
df.groupby("diagnosis", observed=True)[two].agg(
    ["median", "mean", "std", "max"]).T.round(4)

**과제 3.** 동료가 셀 22의 분할 결과에서 p < 0.05인 변수만 골라 `multipletests`에 넣고 "Holm 보정 후에도 두 변수가 유의했다"고 썼습니다. 동료의 계산을 재현하고, 무엇이 틀렸는지 설명하세요. 셀 23처럼 분할을 1000번 되풀이해 두 방법의 가족단위 오류율을 비교하세요.

### 셀 27. 과제 3 정답. 유의한 p만 골라 보정하면

In [ ]:
p_sig = p_null[p_null < 0.05]                  # 동료: 유의한 p만 골라서
print("colleague:", p_sig.round(4), "->",
      multipletests(p_sig, method="holm")[1].round(4))
print("all 30   :", multipletests(p_null, method="holm")[0].sum(),
      "significant")

wrong, right = [], []
for i in range(1000):                          # 셀 23과 같은 분할 1000개
    a = ben.sample(frac=0.5, random_state=i)
    b = ben.drop(a.index)
    p = stats.ttest_ind(a[features], b[features],
                        equal_var=False).pvalue
    sig = p[p < 0.05]
    wrong.append(len(sig) > 0 and multipletests(sig, method="holm")[0].any())
    right.append(multipletests(p, method="holm")[0].any())
print("FWER, only p < 0.05 adjusted:", np.mean(wrong))
print("FWER, all 30 adjusted       :", np.mean(right))